# 06 · Robustness and changes in tactic use

This notebook brings together the evidence on whether associations between tactics persist as tactic use changes. It uses the full-period estimates from notebook 03, the annual comparisons from notebook 04, and the model fits and calibrated departures from notebook 05.

1. Compare the four sampling chains to measure Monte Carlo uncertainty in annual correlations.
2. Summarize the sample, sampler diagnostics, annual similarity and model fit under each reference.
3. Identify the tactic pairs contributing to the largest departures in years flagged by the calibrated quasi-$F$ comparison.
4. Compare changes in tactic prevalence, pair shares and pair associations between 2008–2018 and 2021–2024.
5. Estimate annual co-occurrence ratios between codebook families using the same day-block bootstrap samples.

The results are saved as aggregate tables and displayed in their producing cells. Action-level and day-level inputs are read from the local files generated by notebooks 01 and 02.


In [1]:
import os

for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(k, "2")
import json, itertools
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
CFG = json.loads((ROOT / "config" / "analysis.json").read_text())
TAB = ROOT / "results" / "tables"
AGG = ROOT / "results" / "aggregates"


def display_and_export_table(df, name, index=False, float_format="%.4f"):
    display(df)
    df.to_csv(TAB / f"{name}.csv", index=index)
    df.to_latex(
        TAB / f"{name}.tex", index=index, escape=True, float_format=lambda x: float_format % x
    )
    return df


PAIRS = np.array(list(itertools.combinations(range(1, 36), 2)))
YEARS = np.arange(2008, 2025)
T = len(YEARS)
SPECS = ["N1", "N2", "N2_clean", "N2_allmedia", "N2_order", "N1_k2"]
elig = np.load(AGG / "eligible_primary.npy")
lr = lambda O, E: np.log2((O + 0.5) / (E + 0.5))


## 1. How much do annual correlations vary across sampling chains?

Each chain supplies an independent estimate of the reference counts. Recomputing the annual correlations from each chain measures Monte Carlo uncertainty; the day-block resampling in notebook 04 addresses sampling uncertainty in the protest record.


In [2]:
rows = []
for name in SPECS:
    z = np.load(AGG / f"null_{name}.npz")
    C = z["counts"].astype(float)
    ch = z["chain"]
    Oy = z["observed_year"].astype(float)
    sd_all = C.std(0, ddof=1)
    tot = C.sum(1)
    per_chain = []
    for c in np.unique(ch):
        Ey = C[ch == c].mean(0)
        r = []
        for ti in range(T):
            Er = Ey.sum(0) - Ey[ti]
            sdr = (tot - C[:, ti, :]).std(0, ddof=1)
            m = elig & (Ey[ti] >= 5) & (Er >= 5) & (sd_all[ti] > 0) & (sdr > 0)
            r.append(
                pearsonr(lr(Oy[ti], Ey[ti])[m], lr(Oy.sum(0) - Oy[ti], Er)[m])[0]
                if m.sum() >= 3
                else np.nan
            )
        per_chain.append(r)
    per_chain = np.array(per_chain)
    for ti, yr in enumerate(YEARS):
        rows.append(
            {
                "reference": name,
                "year": yr,
                "pearson_chain_min": np.nanmin(per_chain[:, ti]),
                "pearson_chain_max": np.nanmax(per_chain[:, ti]),
                "pearson_chain_range": np.nanmax(per_chain[:, ti]) - np.nanmin(per_chain[:, ti]),
            }
        )
mc = display_and_export_table(pd.DataFrame(rows), "monte_carlo_error_year_vs_rest")
mc.groupby("reference").pearson_chain_range.agg(["median", "max"]).round(4)


,reference,year,pearson_chain_min,pearson_chain_max,pearson_chain_range
0,N1,2008,0.749713,0.769032,0.019319
1,N1,2009,0.852287,0.863992,0.011706
2,N1,2010,0.903291,0.908351,0.005060
3,N1,2011,0.888251,0.892361,0.004110
4,N1,2012,0.890740,0.895241,0.004500
...,...,...,...,...,...
97,N1_k2,2020,0.847387,0.850975,0.003588
98,N1_k2,2021,0.871988,0.875819,0.003831
99,N1_k2,2022,0.845655,0.861471,0.015816
100,N1_k2,2023,0.685462,0.691117,0.005655


,median,max
reference,,
N1,0.0045,0.0193
N1_k2,0.0054,0.0174
N2,0.0044,0.1015
N2_allmedia,0.0095,0.0820
N2_clean,0.0082,0.2012
N2_order,0.0097,0.0739


## 2. How do the findings compare across reference models?

The summary uses the same annual departure criterion as the manuscript: approximate Holm-adjusted $p<0.05$ in the quasi-$F$ comparison with within-year variation. The two fit gains compare M1 with MA using residual deviance and dispersion-adjusted Pearson discrepancy. The permutation-adjusted quantity and heterogeneity measures are calibration diagnostics described in the Supplementary Methods.


In [3]:
glob = pd.read_csv(TAB / "global_structure_tests.csv").set_index("reference")
yvr = pd.read_csv(TAB / "annual_year_vs_rest.csv")
period = pd.read_csv(TAB / "period_comparison.csv").set_index("reference")
mcmc = pd.read_csv(TAB / "mcmc_diagnostics.csv").set_index("spec")
dec = pd.read_csv(TAB / "loglinear_decomposition.csv").set_index("reference")
het = pd.read_csv(TAB / "heterogeneity_summary.csv").set_index("reference")
calib = pd.read_csv(TAB / "annual_calibration.csv")
phit = pd.read_csv(TAB / "unidiff_phi_by_year.csv")
Ftab = pd.read_csv(TAB / "annual_departure_F.csv")
rows = []
for name in SPECS:
    y = yvr[yvr.reference == name]
    row = {
        "reference": name,
        "actions": int(mcmc.loc[name, "actions"]),
        "multi_tactic_actions": int(mcmc.loc[name, "multi_tactic_actions"]),
        "acceptance": mcmc.loc[name, "acceptance_mean"],
        "changed_fields": mcmc.loc[name, "changed_fields_mean"],
        "fields_in_fixed_strata": mcmc.loc[name, "fields_in_fixed_strata"],
        "rhat_max": mcmc.loc[name, "rhat_max"],
        "evaluable_pairs_full_period": int(glob.loc[name, "evaluable_pairs"]),
        "global_excess_ratio": glob.loc[name, "excess_ratio"],
        "yvr_pearson_median": y.pearson.median(),
        "yvr_pearson_min": y.pearson.min(),
        "yvr_year_min": int(y.loc[y.pearson.idxmin(), "year"]),
        "yvr_strong_sign_median": y.strong_sign_agreement.median(),
        "yvr_pairs_median": y.pairs.median(),
        "period_pearson": period.loc[name, "pearson"],
    }
    if name in calib.reference.unique():
        c = calib[calib.reference == name]
        row.update(
            disattenuated_median=c.disattenuated.median(), disattenuated_min=c.disattenuated.min()
        )
    if name in Ftab.reference.unique():
        f = Ftab[Ftab.reference == name]
        flagged_years = f.loc[f.p_holm < 0.05, "year"].astype(int)
        row.update(
            years_beyond_noise_count=len(flagged_years),
            years_beyond_noise_list=",".join(map(str, flagged_years)),
        )
    if name in dec.index:
        row.update(
            fit_gain_deviance=dec.loc[name, "fit_gain_deviance"],
            fit_gain_dispersion_adjusted=dec.loc[name, "fit_gain_dispersion_adjusted"],
            fit_gain_permutation_adjusted=dec.loc[name, "fit_gain_permutation_adjusted"],
            share_structure_constant=dec.loc[name, "share_structure_constant"],
            phi_min=phit[phit.reference == name].phi.min(),
            phi_max=phit[phit.reference == name].phi.max(),
        )
    if name in het.index:
        row.update(pooled_I2=het.loc[name, "pooled_I2"], median_I2=het.loc[name, "median_I2"])
    rows.append(row)
robust = display_and_export_table(pd.DataFrame(rows), "robustness_summary")
robust.set_index("reference").T


,reference,actions,multi_tactic_actions,acceptance,changed_fields,fields_in_fixed_strata,rhat_max,evaluable_pairs_full_period,global_excess_ratio,yvr_pearson_median,...,years_beyond_noise_count,years_beyond_noise_list,fit_gain_deviance,fit_gain_dispersion_adjusted,fit_gain_permutation_adjusted,share_structure_constant,phi_min,phi_max,pooled_I2,median_I2
0,N1,27488,13405,0.620255,0.871467,0.000000,1.013231,291,93.601083,0.845288,...,9.0,"2011,2013,2015,2016,2018,2019,2020,2021,2022",0.615651,0.871696,0.856048,0.758438,0.690998,1.242664,0.801769,0.399907
1,N2,27488,13405,0.584209,0.738566,0.008960,1.011511,274,49.716001,0.832855,...,4.0,"2016,2019,2020,2021",0.712438,0.959420,1.168779,0.689854,0.800651,1.152488,0.623234,0.040706
2,N2_clean,23559,6666,0.614812,0.730035,0.011802,1.009244,209,37.083273,0.871918,...,2.0,"2020,2021",0.655319,0.890779,1.252137,0.681496,0.872864,1.353652,0.610103,0.000000
3,N2_allmedia,34118,15863,0.597585,0.747832,0.007040,1.007844,280,55.695549,0.841709,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,N2_order,27488,13405,0.382012,0.686546,0.008960,1.013209,272,38.653124,0.815327,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,N1_k2,8981,8981,0.554476,0.836808,0.000000,1.007306,142,51.247772,0.848068,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


reference,N1,N2,N2_clean,N2_allmedia,N2_order,N1_k2
actions,27488,27488,23559,34118,27488,8981
multi_tactic_actions,13405,13405,6666,15863,13405,8981
acceptance,0.620255,0.584209,0.614812,0.597585,0.382012,0.554476
changed_fields,0.871467,0.738566,0.730035,0.747832,0.686546,0.836808
fields_in_fixed_strata,0.0,0.00896,0.011802,0.00704,0.00896,0.0
rhat_max,1.013231,1.011511,1.009244,1.007844,1.013209,1.007306
evaluable_pairs_full_period,291,274,209,280,272,142
global_excess_ratio,93.601083,49.716001,37.083273,55.695549,38.653124,51.247772
yvr_pearson_median,0.845288,0.832855,0.871918,0.841709,0.815327,0.848068
yvr_pearson_min,0.750428,0.697198,0.762689,0.736038,0.657368,0.688083


## 3. Which pairs contribute to the flagged departures?

For years whose departure from the common pattern exceeds within-year noise under N2, rank the eligible pairs by their Pearson residual, $(O_{ht}-\mu_{ht})/\sqrt{\mu_{ht}}$. The prediction combines the year's reference expectation with the pair association fitted to the other 16 years. Ranking requires $E_{ht}\ge2$ and at least one occurrence in the other years. These selected pairs describe the departures identified by the annual comparison in notebook 05.


In [4]:
flag = Ftab[(Ftab.reference == "N2") & (Ftab.p_holm < 0.05)].year.tolist()
z = np.load(AGG / "null_N2.npz")
H = np.where(elig)[0]
O = z["observed_year"][:, H].astype(float)
E = z["counts"].mean(0)[:, H]
dic = pd.read_csv(TAB / "tactic_dictionary.csv").set_index("code")


def fit_M1(O, E, tol=1e-10, max_iter=5000):
    """Fit year and pair effects with reference expectations as a Poisson offset."""
    a = np.ones((O.shape[0], 1))
    b = np.ones((1, O.shape[1]))
    for it in range(max_iter):
        nb_, db_ = O.sum(0, keepdims=True), (E * a).sum(0, keepdims=True)
        b_new = np.divide(nb_, db_, out=np.zeros_like(nb_, dtype=float), where=db_ > 0)
        na_, da_ = O.sum(1, keepdims=True), (E * b_new).sum(1, keepdims=True)
        a_new = np.divide(na_, da_, out=np.ones_like(na_, dtype=float), where=da_ > 0)
        done = np.nanmax(np.abs(a_new - a)) < tol and np.nanmax(np.abs(b_new - b)) < tol
        a, b = a_new, b_new
        if done:
            break
    return E * a * b, a.ravel(), b.ravel()


rows = []
for yr in flag:
    ti = yr - 2008
    others = [s for s in range(T) if s != ti]
    _, _, b_o = fit_M1(O[others], E[others])
    cells = (E[ti] >= 2) & (b_o > 0)
    mu = E[ti] * b_o
    mu = mu * O[ti, cells].sum() / mu[cells].sum()
    res = np.where(cells, (O[ti] - mu) / np.sqrt(np.where(mu > 0, mu, 1)), np.nan)
    # Rank only finite residuals among eligible pairs, retaining the sign of each departure.
    valid = np.flatnonzero(np.isfinite(res))
    order = valid[np.argsort(res[valid])]
    for kind, idx in [
        ("more_than_common_structure", [i for i in order[::-1] if res[i] > 0][:8]),
        ("fewer_than_common_structure", [i for i in order if res[i] < 0][:8]),
    ]:
        for i in idx:
            h = H[i]
            rows.append(
                {
                    "year": yr,
                    "direction": kind,
                    "code_i": PAIRS[h, 0],
                    "code_j": PAIRS[h, 1],
                    "label_i": dic.loc[PAIRS[h, 0], "label_en"],
                    "label_j": dic.loc[PAIRS[h, 1], "label_en"],
                    "observed": int(O[ti, i]),
                    "predicted_common_structure": mu[i],
                    "ratio": O[ti, i] / mu[i],
                    "pearson_residual": res[i],
                }
            )
dep = display_and_export_table(pd.DataFrame(rows), "departure_pairs_flagged_years_N2")
print("flagged years (N2):", flag)


,year,direction,code_i,code_j,label_i,label_j,observed,predicted_common_structure,ratio,pearson_residual
0,2016,more_than_common_structure,2,19,Static demonstration,Building occupation,30,12.149460,2.469246,5.121213
1,2016,more_than_common_structure,7,10,Press conference or statement,Chants or slogans,11,3.867078,2.844525,3.627238
2,2016,more_than_common_structure,2,22,Static demonstration,Roadblock or barricade,94,65.826526,1.427996,3.472481
3,2016,more_than_common_structure,1,11,March,Boycott,7,2.078664,3.367548,3.413429
4,2016,more_than_common_structure,18,27,Strike or stoppage,Arson,1,0.074679,13.390720,3.386059
...,...,...,...,...,...,...,...,...,...,...
59,2021,fewer_than_common_structure,2,10,Static demonstration,Chants or slogans,371,411.645755,0.901260,-2.003334
60,2021,fewer_than_common_structure,22,33,Roadblock or barricade,Attack on police or security forces,86,106.548023,0.807148,-1.990661
61,2021,fewer_than_common_structure,2,34,Static demonstration,Attack on bystanders,1,5.670879,0.176340,-1.961433
62,2021,fewer_than_common_structure,27,29,Arson,Looting,2,6.408167,0.312102,-1.741370


flagged years (N2): [2016, 2019, 2020, 2021]


## 4. Did tactic prevalence and pair associations change by similar amounts?

Compare 2008–2018 with 2021–2024 on a common multiplicative scale (Fig. 1). For each tactic, change is the log2 ratio of its share of actions in the later period to its share in the earlier period. Pairs are selected using frequencies and expected counts from 2008–2018, and must have an expected count of at least 5 and a varying reference count in both periods.

For each pair, the observed change in its share of all pair occurrences ($y$) is the expected change under the reference ($x$) plus the change in its association, $\Delta\theta=y-x$. Each reference preserves the number of tactics per action, so observed and expected totals of pair occurrences are equal. The calculation adds 0.5 to each pair or tactic count before taking logarithms, as specified in the Supplementary Methods.

Intervals come from 999 resamples of protest days within years, with per-day expectations held fixed. The same bootstrap samples supply the annual family ratios in section 5. Annual comparisons additionally contrast each year with the other 16 years pooled, covering the complete 2008–2024 record.


In [5]:
LOCAL = ROOT / "data_local"
B_CHG, CHG_SEED = CFG["change"]["B"], CFG["change"]["seed"]
periods = CFG["change"]["periods"]
EARLY = (YEARS >= periods[0][0]) & (YEARS <= periods[0][1])
LATE = (YEARS >= periods[1][0]) & (YEARS <= periods[1][1])
REFS_CHG = ["N2", "N1"]
tdic = pd.read_csv(TAB / "tactic_dictionary.csv").set_index("code")
acts = pd.read_parquet(LOCAL / "actions.parquet")
acts = acts[acts.eligible & (acts.pb == 0)].reset_index(drop=True)  # main sample: newspaper actions
slots = acts[[f"s{i}" for i in range(1, 7)]].to_numpy()
ind = np.zeros((len(acts), 36))
for k in range(6):
    ok = (slots[:, k] >= 1) & (slots[:, k] <= 35)
    ind[np.where(ok)[0], slots[ok, k]] = 1
ind = ind[:, 1:]  # actions x 35 tactic indicators
udays, dinv = np.unique(acts.day.astype(str).to_numpy(), return_inverse=True)
year_day = pd.to_datetime(udays).year.to_numpy()
day_sets = [np.where(year_day == y)[0] for y in YEARS]
tac_day = np.zeros((len(udays), 35))
np.add.at(tac_day, dinv, ind)
act_day = np.bincount(dinv, minlength=len(udays)).astype(float)
counts_year = pd.read_csv(TAB / "tactic_counts_by_year.csv").set_index("year")
counts_year.columns = counts_year.columns.astype(int)
cov = pd.read_csv(TAB / "annual_coverage.csv")
cov = cov[cov["sample"] == "newspapers"].set_index("year")
assert len(acts) == cov.actions.sum()
assert np.allclose([act_day[s].sum() for s in day_sets], cov.loc[YEARS, "actions"])
assert np.allclose(
    [tac_day[s].sum(0) for s in day_sets], counts_year.loc[YEARS, list(range(1, 36))].to_numpy()
)
keep = [c for c in range(1, 36) if counts_year[c].sum() >= 50]
kidx = np.array(keep) - 1


def day_arrays(name):
    z = np.load(LOCAL / f"day_OE_{name}.npz", allow_pickle=True)
    days = z["days"].astype(str)
    pos = np.searchsorted(udays, days)
    assert (udays[pos] == days).all()
    O = np.zeros((len(udays), 595))
    E = np.zeros((len(udays), 595))
    O[pos] = z["observed"]
    E[pos] = z["expected"]
    return O, E


dayOE = {n: day_arrays(n) for n in REFS_CHG}
eligible_early = np.load(AGG / "eligible_early_period.npy")
masks = {}
for n in REFS_CHG:
    z = np.load(AGG / f"null_{n}.npz")
    C = z["counts"].astype(float)
    Oy = z["observed_year"].astype(float)
    Ea, Eb = C[:, EARLY].sum(1).mean(0), C[:, LATE].sum(1).mean(0)
    sda, sdb = C[:, EARLY].sum(1).std(0, ddof=1), C[:, LATE].sum(1).std(0, ddof=1)
    masks[n] = (
        eligible_early & (Ea >= 5) & (Eb >= 5) & (sda > 0) & (sdb > 0)
    )  # the pairs of the two-block comparison
    O, E = dayOE[n]
    assert np.allclose([O[s].sum(0) for s in day_sets], Oy)
    assert np.allclose([E[s].sum(0) for s in day_sets], C.mean(0), rtol=1e-4, atol=1e-3)
    assert np.allclose(O.sum(1), E.sum(1))  # list lengths fixed: equal pair totals per day
print(
    "actions",
    len(acts),
    "| days",
    len(udays),
    "| tactics",
    len(keep),
    "| pairs",
    {n: int(m.sum()) for n, m in masks.items()},
)


def changes(tac, act, O, E, m):
    """Late-vs-early log2 changes from block totals: tactic shares, pair share (y), expected pair share (x), association (y - x)."""
    d_tac = np.log2(((tac["L"][kidx] + 0.5) / act["L"]) / ((tac["E"][kidx] + 0.5) / act["E"]))
    tot = {k: O[k].sum() for k in "EL"}
    y = np.log2((O["L"][m] + 0.5) / tot["L"]) - np.log2((O["E"][m] + 0.5) / tot["E"])
    x = np.log2((E["L"][m] + 0.5) / tot["L"]) - np.log2((E["E"][m] + 0.5) / tot["E"])
    return d_tac, y, x, y - x


blk = lambda arr, sel: arr[np.isin(year_day, YEARS[sel])].sum(0)
tac0 = {"E": blk(tac_day, EARLY), "L": blk(tac_day, LATE)}
act0 = {"E": blk(act_day, EARLY), "L": blk(act_day, LATE)}
OE0 = {
    n: (
        {"E": blk(dayOE[n][0], EARLY), "L": blk(dayOE[n][0], LATE)},
        {"E": blk(dayOE[n][1], EARLY), "L": blk(dayOE[n][1], LATE)},
    )
    for n in REFS_CHG
}
est = {n: changes(tac0, act0, *OE0[n], masks[n]) for n in REFS_CHG}

FAMS = [
    "A Peaceful I",
    "B Peaceful II",
    "C Artistic",
    "D Disruptive",
    "E Self-destructive",
    "F Violent",
]
fam_of = tdic.loc[list(range(1, 36)), "family_label"].to_numpy()
fam_pairs = [(a, b) for i, a in enumerate(FAMS) for b in FAMS[i:]]
Fmat = np.zeros((595, len(fam_pairs)))
for h, (ci, cj) in enumerate(PAIRS):
    fa, fb = sorted([fam_of[ci - 1], fam_of[cj - 1]], key=FAMS.index)
    Fmat[h, fam_pairs.index((fa, fb))] = 1

rng = np.random.default_rng(CHG_SEED)
boot_tac = np.zeros((B_CHG, len(keep)))
boot_dth = {n: np.zeros((B_CHG, int(masks[n].sum()))) for n in REFS_CHG}
boot_fam = {n: np.zeros((B_CHG, T, len(fam_pairs), 2)) for n in REFS_CHG}
for b in range(B_CHG):
    wy = [
        np.bincount(rng.integers(0, len(s), len(s)), minlength=len(s)).astype(float)
        for s in day_sets
    ]
    tac_y = np.array([w @ tac_day[s] for w, s in zip(wy, day_sets)])
    act_y = np.array([w @ act_day[s] for w, s in zip(wy, day_sets)])
    tb = {"E": tac_y[EARLY].sum(0), "L": tac_y[LATE].sum(0)}
    ab = {"E": act_y[EARLY].sum(), "L": act_y[LATE].sum()}
    for n in REFS_CHG:
        O, E = dayOE[n]
        Oy_b = np.array([w @ O[s] for w, s in zip(wy, day_sets)])
        Ey_b = np.array([w @ E[s] for w, s in zip(wy, day_sets)])
        d_tac, _, _, dth = changes(
            tb,
            ab,
            {"E": Oy_b[EARLY].sum(0), "L": Oy_b[LATE].sum(0)},
            {"E": Ey_b[EARLY].sum(0), "L": Ey_b[LATE].sum(0)},
            masks[n],
        )
        boot_dth[n][b] = dth
        boot_fam[n][b, :, :, 0] = Oy_b @ Fmat
        boot_fam[n][b, :, :, 1] = Ey_b @ Fmat
    boot_tac[b] = d_tac
q95 = lambda a: (np.quantile(a, 0.025, axis=0), np.quantile(a, 0.975, axis=0))

lo, hi = q95(boot_tac)
tac_tab = pd.DataFrame(
    {
        "code": keep,
        "label": [tdic.loc[c, "label_en"] for c in keep],
        "family": fam_of[kidx],
        "actions_2008_2018": tac0["E"][kidx].astype(int),
        "actions_2021_2024": tac0["L"][kidx].astype(int),
        "share_2008_2018": tac0["E"][kidx] / act0["E"],
        "share_2021_2024": tac0["L"][kidx] / act0["L"],
        "log2_change": est["N2"][0],
        "lo": lo,
        "hi": hi,
    }
)
tac_tab["beyond_noise"] = (tac_tab.lo > 0) | (tac_tab.hi < 0)
tac_tab = display_and_export_table(tac_tab, "change_tactic_shares_blocks")

pair_rows, summ = [], []
for n in REFS_CHG:
    d_tac, y, x, dth = est[n]
    lo, hi = q95(boot_dth[n])
    hs = np.where(masks[n])[0]
    O, E = OE0[n]
    beyond = (lo > 0) | (hi < 0)
    big = np.abs(y) >= 1
    for k, h in enumerate(hs):
        pair_rows.append(
            {
                "reference": n,
                "code_i": PAIRS[h, 0],
                "code_j": PAIRS[h, 1],
                "label_i": tdic.loc[PAIRS[h, 0], "label_en"],
                "label_j": tdic.loc[PAIRS[h, 1], "label_en"],
                "observed_2008_2018": O["E"][h],
                "observed_2021_2024": O["L"][h],
                "expected_2008_2018": E["E"][h],
                "expected_2021_2024": E["L"][h],
                "share_change_observed": y[k],
                "share_change_expected": x[k],
                "association_change": dth[k],
                "lo": lo[k],
                "hi": hi[k],
                "beyond_noise": beyond[k],
            }
        )
    summ.append(
        {
            "reference": n,
            "tactics": len(keep),
            "tactics_median_abs_change": np.median(np.abs(d_tac)),
            "tactics_beyond_noise": int(tac_tab.beyond_noise.sum()),
            "pairs": len(hs),
            "pairs_median_abs_share_change": np.median(np.abs(y)),
            "pairs_median_abs_association_change": np.median(np.abs(dth)),
            "pairs_association_beyond_noise": int(beyond.sum()),
            "r_expected_observed": np.corrcoef(x, y)[0, 1],
            "reduction_vs_unchanged_share": 1 - np.sum((y - x) ** 2) / np.sum(y**2),
            "pairs_doubled_or_halved": int(big.sum()),
            "doubled_or_halved_in_expected_direction": int(
                (np.sign(x[big]) == np.sign(y[big])).sum()
            ),
        }
    )
pair_tab = display_and_export_table(pd.DataFrame(pair_rows), "change_pairs_blocks")
change_summary = display_and_export_table(pd.DataFrame(summ), "change_summary")

yr_rows = []
for n in REFS_CHG:
    z = np.load(AGG / f"null_{n}.npz")
    C = z["counts"].astype(float)
    Oy = z["observed_year"].astype(float)
    Ey, sdy, tot = C.mean(0), C.std(0, ddof=1), C.sum(1)
    for ti, yr in enumerate(YEARS):
        Or, Er = Oy.sum(0) - Oy[ti], Ey.sum(0) - Ey[ti]
        sdr = (tot - C[:, ti]).std(0, ddof=1)
        m = elig & (Ey[ti] >= 5) & (Er >= 5) & (sdy[ti] > 0) & (sdr > 0)  # the year-vs-rest pairs
        ty, tr = Oy[ti].sum(), Or.sum()
        y = np.log2((Oy[ti][m] + 0.5) / ty) - np.log2((Or[m] + 0.5) / tr)
        x = np.log2((Ey[ti][m] + 0.5) / ty) - np.log2((Er[m] + 0.5) / tr)
        yr_rows.append(
            {
                "reference": n,
                "year": yr,
                "pairs": int(m.sum()),
                "r_expected_observed": np.corrcoef(x, y)[0, 1],
                "reduction_vs_unchanged_share": 1 - np.sum((y - x) ** 2) / np.sum(y**2),
                "median_abs_share_change": np.median(np.abs(y)),
                "median_abs_association_change": np.median(np.abs(y - x)),
            }
        )
change_by_year = display_and_export_table(pd.DataFrame(yr_rows), "change_by_year")
change_by_year.groupby("reference")[
    ["r_expected_observed", "reduction_vs_unchanged_share"]
].describe().round(2)


actions 27488 | days 5207 | tactics 32 | pairs {'N2': 108, 'N1': 113}


,code,label,family,actions_2008_2018,actions_2021_2024,share_2008_2018,share_2021_2024,log2_change,lo,hi,beyond_noise
0,1,March,A Peaceful I,4419,802,0.256055,0.162942,-0.651361,-0.855633,-0.455844,True
1,2,Static demonstration,A Peaceful I,3644,1504,0.211148,0.305567,0.533512,0.414874,0.650170,True
2,3,Rally with speakers,A Peaceful I,518,80,0.030015,0.016254,-0.877335,-1.451125,-0.439118,True
3,4,Caravan,A Peaceful I,258,99,0.014950,0.020114,0.432553,-0.132619,0.973236,False
4,5,Petition or fund collection,A Peaceful I,495,158,0.028682,0.032101,0.165547,-0.154404,0.441720,False
5,7,Press conference or statement,B Peaceful II,41,6,0.002376,0.001219,-0.864651,-2.371717,0.248499,False
6,8,Assembly or open debate,B Peaceful II,35,4,0.002028,0.000813,-1.169873,-2.971697,-0.036360,True
7,9,Commemoration,B Peaceful II,274,175,0.015877,0.035555,1.164614,0.376114,1.845005,True
8,10,Chants or slogans,B Peaceful II,3608,2224,0.209062,0.451849,1.112031,1.004132,1.217594,True
9,11,Boycott,B Peaceful II,28,40,0.001622,0.008127,2.316909,-0.044135,4.092018,False


,reference,code_i,code_j,label_i,label_j,observed_2008_2018,observed_2021_2024,expected_2008_2018,expected_2021_2024,share_change_observed,share_change_expected,association_change,lo,hi,beyond_noise
0,N2,1,2,March,Static demonstration,226.0,57.0,379.973973,103.396396,-0.819088,-0.713863,-0.105225,-0.571824,0.320840,False
1,N2,1,3,March,Rally with speakers,390.0,49.0,129.920921,15.286286,-1.821033,-1.887642,0.066609,-0.632964,0.581757,False
2,N2,1,4,March,Caravan,41.0,13.0,34.449449,6.608609,-0.461363,-1.138841,0.677479,-0.352382,1.485964,False
3,N2,1,5,March,Petition or fund collection,218.0,60.0,99.193193,27.331331,-0.693837,-0.681996,-0.011841,-0.416317,0.383717,False
4,N2,1,9,March,Commemoration,81.0,83.0,35.534535,21.262262,1.193765,0.431238,0.762528,-0.124873,1.433384,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,N1,27,34,Arson,Attack on bystanders,57.0,95.0,6.323323,24.358358,1.890728,3.023974,-1.133246,-1.626273,-0.642590,True
217,N1,28,30,Property destruction,Explosives or firearms,79.0,47.0,12.216216,17.429429,0.415762,1.654449,-1.238687,-1.802668,-0.681903,True
218,N1,28,33,Property destruction,Attack on police or security forces,200.0,84.0,32.451451,16.759760,-0.087790,0.225860,-0.313649,-0.668123,0.010682,False
219,N1,30,33,Explosives or firearms,Attack on police or security forces,241.0,157.0,25.901902,38.980981,0.542118,1.739305,-1.197187,-1.575660,-0.831106,True


,reference,tactics,tactics_median_abs_change,tactics_beyond_noise,pairs,pairs_median_abs_share_change,pairs_median_abs_association_change,pairs_association_beyond_noise,r_expected_observed,reduction_vs_unchanged_share,pairs_doubled_or_halved,doubled_or_halved_in_expected_direction
0,N2,32,0.870993,21,108,0.862744,0.393862,17,0.871973,0.737206,50,49
1,N1,32,0.870993,21,113,0.956368,0.649848,37,0.794306,0.606458,55,49


,reference,year,pairs,r_expected_observed,reduction_vs_unchanged_share,median_abs_share_change,median_abs_association_change
0,N2,2008,61,0.826667,0.666659,0.624463,0.366147
1,N2,2009,51,0.804263,0.676710,0.845139,0.348606
2,N2,2010,32,0.649789,0.369406,0.747021,0.354302
3,N2,2011,84,0.853140,0.750951,0.786919,0.333299
4,N2,2012,62,0.682998,0.479959,0.683761,0.405724
5,N2,2013,66,0.815726,0.698023,0.636241,0.272741
6,N2,2014,60,0.791488,0.652453,0.564532,0.323425
7,N2,2015,32,0.502350,0.339040,0.481757,0.338824
8,N2,2016,66,0.785361,0.647855,0.797105,0.366731
9,N2,2017,41,0.783582,0.661915,0.517423,0.353997


r_expected_observed                                            \
                        count  mean   std   min   25%   50%   75%   max   
reference                                                                 
N1                       17.0  0.68  0.13  0.32  0.65  0.70  0.75  0.85   
N2                       17.0  0.77  0.09  0.50  0.75  0.79  0.82  0.87   

          reduction_vs_unchanged_share                                      \
                                 count  mean   std   min   25%   50%   75%   
reference                                                                    
N1                                17.0  0.47  0.14  0.15  0.44  0.46  0.56   
N2                                17.0  0.61  0.13  0.34  0.56  0.66  0.70   

                 
            max  
reference        
N1         0.69  
N2         0.75

## 5. Did affinities and avoidances between families recur every year?

For each year and pair of codebook families, pool observed and expected counts across all tactic pairs linking the two families before calculating $(O+0.5)/(E+0.5)$. Intervals use the day-block bootstrap samples from section 4. The comparison with the full-period family table verifies that the annual counts sum to the same totals. Figure 3a displays six family pairs; the output table includes all 21 within-family and between-family combinations under N2 and N1.


In [6]:
fam_rows = []
for n in REFS_CHG:
    O, E = dayOE[n]
    Oy_f = np.array([O[s].sum(0) for s in day_sets]) @ Fmat
    Ey_f = np.array([E[s].sum(0) for s in day_sets]) @ Fmat
    rb = (boot_fam[n][..., 0] + 0.5) / (boot_fam[n][..., 1] + 0.5)
    lo_b, hi_b = q95(rb)
    for k, (fa, fb) in enumerate(fam_pairs):
        for ti, yr in enumerate(YEARS):
            fam_rows.append(
                {
                    "reference": n,
                    "family_a": fa,
                    "family_b": fb,
                    "year": yr,
                    "observed": Oy_f[ti, k],
                    "expected": Ey_f[ti, k],
                    "ratio": (Oy_f[ti, k] + 0.5) / (Ey_f[ti, k] + 0.5),
                    "lo": lo_b[ti, k],
                    "hi": hi_b[ti, k],
                }
            )
fam_year = display_and_export_table(pd.DataFrame(fam_rows), "family_ratio_by_year")
full = pd.read_csv(TAB / "family_structure.csv")
chk = (
    fam_year.groupby(["reference", "family_a", "family_b"])[["observed", "expected"]]
    .sum()
    .reset_index()
)
chk = chk.merge(full, on=["reference", "family_a", "family_b"], suffixes=("", "_full"))
assert np.allclose(chk.observed, chk.observed_full) and np.allclose(
    chk.expected, chk.expected_full, rtol=1e-4, atol=1e-2
)
HEAD = [
    ("C Artistic", "D Disruptive"),
    ("C Artistic", "F Violent"),
    ("B Peaceful II", "D Disruptive"),
    ("A Peaceful I", "B Peaceful II"),
    ("A Peaceful I", "C Artistic"),
    ("A Peaceful I", "D Disruptive"),
]
f2 = fam_year[fam_year.reference == "N2"].set_index(["family_a", "family_b"])
pd.DataFrame(
    [
        {
            "families": f"{a} / {b}",
            "years_below_1": int((f2.loc[(a, b)].ratio < 1).sum()),
            "years_above_1": int((f2.loc[(a, b)].ratio > 1).sum()),
            "years_interval_below_1": int((f2.loc[(a, b)].hi < 1).sum()),
            "years_interval_above_1": int((f2.loc[(a, b)].lo > 1).sum()),
            "min": f2.loc[(a, b)].ratio.min(),
            "max": f2.loc[(a, b)].ratio.max(),
        }
        for a, b in HEAD
    ]
).round(2)


,reference,family_a,family_b,year,observed,expected,ratio,lo,hi
0,N2,A Peaceful I,A Peaceful I,2008,137.0,97.715715,1.399980,1.144481,1.638546
1,N2,A Peaceful I,A Peaceful I,2009,103.0,83.470470,1.232576,0.988684,1.475959
2,N2,A Peaceful I,A Peaceful I,2010,87.0,73.749750,1.178455,0.931954,1.413980
3,N2,A Peaceful I,A Peaceful I,2011,150.0,121.606607,1.232530,1.019769,1.451571
4,N2,A Peaceful I,A Peaceful I,2012,130.0,70.127127,1.847732,1.441016,2.284042
...,...,...,...,...,...,...,...,...,...
709,N1,F Violent,F Violent,2020,417.0,197.436436,2.109263,1.782938,2.491337
710,N1,F Violent,F Violent,2021,452.0,191.964965,2.351077,1.992387,2.837284
711,N1,F Violent,F Violent,2022,369.0,117.097097,3.142084,2.677602,3.638884
712,N1,F Violent,F Violent,2023,113.0,18.442442,5.991836,4.088151,8.159857


,families,years_below_1,years_above_1,years_interval_below_1,years_interval_above_1,min,max
0,C Artistic / D Disruptive,17,0,15,0,0.06,0.71
1,C Artistic / F Violent,17,0,14,0,0.03,0.92
2,B Peaceful II / D Disruptive,17,0,17,0,0.38,0.75
3,A Peaceful I / B Peaceful II,0,17,0,17,1.31,1.66
4,A Peaceful I / C Artistic,0,17,0,12,1.14,1.87
5,A Peaceful I / D Disruptive,11,6,1,1,0.75,1.20
